# Telco Customer Churn — End-to-End Analysis

This notebook includes:
- Data loading, cleaning, and type conversion
- EDA charts and churn-pattern insights
- Model training + evaluation
- SHAP/ELI5 explainability
- Top churn drivers
- SQL-based business insights
- Refined customer segments (At Risk, Loyal, Dormant)
- Retention strategies and final recommendations


In [ ]:
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style='whitegrid')

BASE_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == 'submission' else Path.cwd().resolve()
DATA_PATH = BASE_DIR / 'data' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
OUTPUT_DIR = BASE_DIR / 'outputs'
FIG_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
FIG_DIR.mkdir(exist_ok=True, parents=True)

print('BASE_DIR:', BASE_DIR)
print('DATA FOUND:', DATA_PATH.exists())

## 1) Load and inspect data

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
display(df_raw.head())
print('Shape:', df_raw.shape)
display(df_raw.dtypes)
display(df_raw.isna().sum().sort_values(ascending=False).head(10))

**Inspection insight:** `TotalCharges` needs numeric conversion; missing values are limited and manageable.

## 2) Cleaning and feature preparation

In [ ]:
df = df_raw.copy()
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

categorical_cols = [c for c in df.select_dtypes(include=['object', 'string']).columns if c != 'Churn']
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode().iloc[0])
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

df['ChurnFlag'] = (df['Churn'].str.lower() == 'yes').astype(int)
print('Missing after clean:', int(df.isna().sum().sum()))
df.to_csv(OUTPUT_DIR / 'cleaned_dataset.csv', index=False)

## 3) EDA charts + insights

In [ ]:
plt.figure(figsize=(6,4))
ax = sns.countplot(data=df, x='Churn', palette='Set2')
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom')
plt.title('Churn distribution')
plt.tight_layout(); plt.savefig(FIG_DIR / 'eda_churn_distribution.png', dpi=200); plt.show()

contract_churn = (df.groupby('Contract')['ChurnFlag'].mean().sort_values(ascending=False)*100).reset_index(name='ChurnRatePct')
plt.figure(figsize=(7,4)); sns.barplot(data=contract_churn, x='Contract', y='ChurnRatePct', palette='viridis')
plt.title('Churn rate by contract (%)'); plt.tight_layout(); plt.savefig(FIG_DIR / 'eda_churn_by_contract.png', dpi=200); plt.show()

plt.figure(figsize=(7,4)); sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette='coolwarm')
plt.title('Monthly charges by churn'); plt.tight_layout(); plt.savefig(FIG_DIR / 'eda_monthly_charges_by_churn.png', dpi=200); plt.show()

plt.figure(figsize=(7,4)); sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, common_norm=False, alpha=0.25)
plt.title('Tenure density by churn'); plt.tight_layout(); plt.savefig(FIG_DIR / 'eda_tenure_density.png', dpi=200); plt.show()

**EDA insights**
- Month-to-month users churn the most.
- High monthly charges are associated with churn.
- Early tenure period is the highest-risk lifecycle window.


## 4) Train model and evaluate

In [ ]:
X = df.drop(columns=['Churn', 'ChurnFlag', 'customerID'], errors='ignore')
y = df['ChurnFlag']

cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

model = RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_split=4, class_weight='balanced', random_state=42, n_jobs=-1)
clf = Pipeline([('preprocessor', preprocessor), ('model', model)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
prob = clf.predict_proba(X_test)[:,1]

metrics = {
    'accuracy': float(accuracy_score(y_test, pred)),
    'precision': float(precision_score(y_test, pred, zero_division=0)),
    'recall': float(recall_score(y_test, pred, zero_division=0)),
    'f1': float(f1_score(y_test, pred, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_test, prob)),
}
print(json.dumps(metrics, indent=2))

cm = confusion_matrix(y_test, pred)
cm_df = pd.DataFrame(cm, index=['Actual 0','Actual 1'], columns=['Pred 0','Pred 1'])
display(cm_df)

plt.figure(figsize=(5,4)); sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues'); plt.title('Confusion matrix')
plt.tight_layout(); plt.savefig(FIG_DIR / 'model_confusion_matrix.png', dpi=200); plt.show()

with open(OUTPUT_DIR / 'model_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

report = pd.DataFrame(classification_report(y_test, pred, output_dict=True)).T
display(report)

## 5) Explainability with SHAP (and ELI5)

In [ ]:
X_sample = X_test.sample(n=min(400, len(X_test)), random_state=42)
y_sample = y_test.loc[X_sample.index]

transformed = clf.named_steps['preprocessor'].transform(X_sample)
if hasattr(transformed, 'toarray'):
    transformed = transformed.toarray()
feature_names = clf.named_steps['preprocessor'].get_feature_names_out().tolist()

status = {'shap': False, 'eli5': False}

try:
    import shap
    explainer = shap.TreeExplainer(clf.named_steps['model'])
    shap_values = explainer.shap_values(transformed)
    vals = shap_values[1] if isinstance(shap_values, list) else shap_values
    vals = np.asarray(vals)

    # Handle binary/multiclass SHAP output shapes robustly
    if vals.ndim == 3:
        # Typical shape: (samples, features, classes)
        class_index = 1 if vals.shape[2] > 1 else 0
        vals = vals[:, :, class_index]
    elif vals.ndim == 1:
        vals = vals.reshape(-1, 1)

    shap_mean = np.abs(vals).mean(axis=0)
    shap_imp = pd.DataFrame({'feature': feature_names, 'mean_abs_shap': shap_mean}).sort_values('mean_abs_shap', ascending=False)
    shap_imp.to_csv(OUTPUT_DIR / 'shap_feature_importance.csv', index=False)

    plt.figure(figsize=(8,6))
    top = shap_imp.head(15).sort_values('mean_abs_shap')
    plt.barh(top['feature'], top['mean_abs_shap'])
    plt.title('Top SHAP drivers (mean |SHAP|)')
    plt.tight_layout(); plt.savefig(FIG_DIR / 'shap_top_drivers.png', dpi=200); plt.show()
    status['shap'] = True
except Exception as e:
    print('SHAP error:', e)

try:
    import eli5
    from eli5.sklearn import PermutationImportance
    perm = PermutationImportance(clf, random_state=42)
    perm.fit(X_sample, y_sample)
    weights = eli5.formatters.as_dataframe.explain_weights_df(eli5.explain_weights(perm, feature_names=X_sample.columns.tolist()))
    if weights is not None and not weights.empty:
        weights.to_csv(OUTPUT_DIR / 'eli5_feature_importance.csv', index=False)
        status['eli5'] = True
except Exception as e:
    print('ELI5 error:', e)

with open(OUTPUT_DIR / 'explainability_status.json', 'w', encoding='utf-8') as f:
    json.dump(status, f, indent=2)
print('Explainability status:', status)


## 6) Top churn drivers

In [ ]:
if (OUTPUT_DIR / 'shap_feature_importance.csv').exists():
    drivers = pd.read_csv(OUTPUT_DIR / 'shap_feature_importance.csv').head(10)
else:
    drivers = pd.DataFrame({'feature': feature_names, 'importance': clf.named_steps['model'].feature_importances_}).sort_values('importance', ascending=False).head(10)

display(drivers)
print('Main churn drivers typically include tenure, contract type, monthly charges, and total charges.')

## 7) Refined customer segmentation: At Risk, Loyal, Dormant

In [ ]:
scored = X.copy()
scored['customerID'] = df['customerID'].values
scored['Contract'] = df['Contract'].values
scored['tenure'] = df['tenure'].values
scored['MonthlyCharges'] = df['MonthlyCharges'].values
scored['ActualChurn'] = df['Churn'].values
scored['churn_probability'] = clf.predict_proba(X)[:,1]

at_risk = (scored['churn_probability'] >= 0.55) | ((scored['Contract']=='Month-to-month') & (scored['tenure'] < 12))
loyal = (scored['churn_probability'] < 0.30) & (scored['tenure'] >= 24) & (scored['Contract'].isin(['One year','Two year']))

scored['segment_label'] = np.select([at_risk, loyal], ['At Risk', 'Loyal'], default='Dormant')

segment_summary = scored.groupby('segment_label').agg(
    customers=('customerID','count'),
    avg_tenure=('tenure','mean'),
    avg_monthly_charges=('MonthlyCharges','mean'),
    avg_churn_prob=('churn_probability','mean'),
    actual_churn_rate=('ActualChurn', lambda s: (s.str.lower()=='yes').mean())
).reset_index()

display(segment_summary)
scored.to_csv(OUTPUT_DIR / 'final_dataset_with_segments.csv', index=False)
segment_summary.to_csv(OUTPUT_DIR / 'segment_summary_refined.csv', index=False)

**Segment logic explained**
- **At Risk**: high predicted churn or short-tenure month-to-month profile.
- **Loyal**: low predicted churn + long tenure + committed contract.
- **Dormant**: moderate/unclear engagement group requiring activation.


## 8) SQL analysis and business insights

In [ ]:
conn = sqlite3.connect(OUTPUT_DIR / 'analysis.db')
scored.to_sql('churn_scored', conn, if_exists='replace', index=False)

q_contract = '''
SELECT Contract, COUNT(*) AS customers,
       ROUND(AVG(churn_probability),4) AS avg_pred_churn,
       ROUND(AVG(CASE WHEN lower(ActualChurn)='yes' THEN 1.0 ELSE 0.0 END),4) AS actual_churn_rate
FROM churn_scored
GROUP BY Contract
ORDER BY actual_churn_rate DESC;
'''

q_segment = '''
SELECT segment_label, COUNT(*) AS customers,
       ROUND(AVG(MonthlyCharges),2) AS avg_monthly_charges,
       ROUND(AVG(churn_probability),4) AS avg_pred_churn,
       ROUND(AVG(CASE WHEN lower(ActualChurn)='yes' THEN 1.0 ELSE 0.0 END),4) AS actual_churn_rate
FROM churn_scored
GROUP BY segment_label
ORDER BY actual_churn_rate DESC;
'''

sql_contract = pd.read_sql_query(q_contract, conn)
sql_segment = pd.read_sql_query(q_segment, conn)
conn.close()

display(sql_contract)
display(sql_segment)

sql_contract.to_csv(OUTPUT_DIR / 'sql_contract_analysis_refined.csv', index=False)
sql_segment.to_csv(OUTPUT_DIR / 'sql_segment_analysis_refined.csv', index=False)


## 9) Business insights, strategies, and final recommendations

### Business insights (why customers churn)
1. Month-to-month contracts have the highest churn concentration.
2. High monthly charges without long-term commitment increase churn propensity.
3. Early-tenure customers are especially sensitive to service-value mismatch.
4. At Risk segment combines both model and behavioral churn signals.

### Actionable retention strategies
- **At Risk**: save-offer bundles, proactive support calls, and first-year nurture journeys.
- **Loyal**: loyalty perks, referral incentives, and premium feature upsell.
- **Dormant**: reactivation campaigns, usage nudges, and plan right-sizing.

### Final recommendations (evaluation-critical)
1. Prioritize month-to-month short-tenure users as first intervention cohort.
2. Deploy weekly churn scoring and connect output to CRM workflows.
3. Track save-rate, churn delta, segment migration, and ARPU uplift per campaign.
4. Use SHAP/ELI5 evidence to align business teams on transparent targeting.


In [ ]:
project_summary = {
    'model_metrics': metrics,
    'artifacts': [
        'outputs/cleaned_dataset.csv',
        'outputs/model_metrics.json',
        'outputs/shap_feature_importance.csv',
        'outputs/final_dataset_with_segments.csv',
        'outputs/segment_summary_refined.csv',
        'outputs/sql_contract_analysis_refined.csv',
        'outputs/sql_segment_analysis_refined.csv'
    ]
}

with open(OUTPUT_DIR / 'project_summary.json', 'w', encoding='utf-8') as f:
    json.dump(project_summary, f, indent=2)

print('Notebook workflow completed. Outputs saved.')